## Describe your model -> fine-tuned LLaMA 2
By Matt Shumer (https://twitter.com/mattshumer_)

The goal of this notebook is to experiment with a new way to make it very easy to build a task-specific model for your use-case.

First, use the best GPU available (go to Runtime -> change runtime type)

To create your model, just go to the first code cell, and describe the model you want to build in the prompt. Be descriptive and clear.

Select a temperature (high=creative, low=precise), and the number of training examples to generate to train the model. From there, just run all the cells.

You can change the model you want to fine-tune by changing `model_name` in the `Define Hyperparameters` cell.

In [ ]:
import os
import sys

try:
    import pyarrow
    from datasets import load_dataset
    import trl
    import peft
    import bitsandbytes
    import accelerate
    import transformers
    print('✅ All libraries are successfully installed and compatible! No restart required.')
except Exception as e:
    print(f'⚠️ Import check failed: {e}')
    print('Installing required libraries...')
    from IPython import get_ipython
    ipython = get_ipython()
    
    # We install WITHOUT the -U flag. This prevents pip from upgrading pre-installed datasets/pyarrow,
    # which keeps the environment stable and avoids binary incompatibility restarts.
    ipython.run_line_magic('pip', 'install -q accelerate peft bitsandbytes transformers trl')
    
    print('Checking compatibility after installation...')
    try:
        import trl
        from datasets import load_dataset
        print('✅ Imports successful! No restart required.')
    except Exception as e2:
        print(f'⚠️ Mismatch detected after install: {e2}')
        print('Upgrading pyarrow and datasets to resolve binary conflicts...')
        ipython.run_line_magic('pip', 'install -q -U pyarrow datasets')
        print()
        print('================================================================================')
        print('🔄 RESTARTING RUNTIME TO APPLY CHANGES...')
        print('The runtime will now restart automatically to apply C-extension updates.')
        print('👉 PLEASE CLICK "Run All" (or "Runtime" -> "Run all") ONE MORE TIME AFTER THE RESTART!')
        print('================================================================================')
        print()
        import time
        time.sleep(2)
        os.kill(os.getpid(), 9)

In [ ]:
import os

# Set base data directory (use Google Drive if in Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = "/content/drive/MyDrive/gpt-llm-trainer"
    os.makedirs(DATA_DIR, exist_ok=True)
    print(f"📁 Google Drive mounted successfully. Data will be saved to: {DATA_DIR}")
except ImportError:
    DATA_DIR = "."
    print(f"📁 Local environment detected. Data will be saved to: {DATA_DIR}")

# Global paths
checkpoint_path = os.path.join(DATA_DIR, "checkpoint_examples.json")
train_path = os.path.join(DATA_DIR, "train.jsonl")
test_path = os.path.join(DATA_DIR, "test.jsonl")

#Data generation step

Write your prompt here. Make it as descriptive as possible!

Then, choose the temperature (between 0 and 1) to use when generating data. Lower values are great for precise tasks, like writing code, whereas larger values are better for creative tasks, like writing stories.

Finally, choose how many examples you want to generate. The more you generate, a) the longer it takes and b) the more expensive data generation will be. But generally, more examples will lead to a higher-quality model. 100 is usually the minimum to start.

In [8]:
prompt = "A model that takes in an unstructured, raw conversation transcript between a doctor and a patient, and outputs a highly professional, structured clinical SOAP note containing Subjective, Objective, Assessment, and Plan sections."
temperature = .4
number_of_examples = 50

Run this to generate the dataset.

In [ ]:
import os
import random
import json
from openai import AzureOpenAI
from tenacity import retry, stop_after_attempt, wait_exponential

try:
    # Try loading from Google Colab Secrets
    from google.colab import userdata
    AZURE_OPENAI_KEY = userdata.get('AZURE_OPENAI_KEY')
    AZURE_OPENAI_ENDPOINT = userdata.get('AZURE_OPENAI_ENDPOINT')
    AZURE_OPENAI_DEPLOYMENT = userdata.get('AZURE_OPENAI_DEPLOYMENT')
except ImportError:
    try:
        # Try loading from Kaggle Secrets
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        AZURE_OPENAI_KEY = user_secrets.get_secret('AZURE_OPENAI_KEY')
        AZURE_OPENAI_ENDPOINT = user_secrets.get_secret('AZURE_OPENAI_ENDPOINT')
        AZURE_OPENAI_DEPLOYMENT = user_secrets.get_secret('AZURE_OPENAI_DEPLOYMENT')
    except ImportError:
        # Fallback to local environment variables
        AZURE_OPENAI_KEY = os.getenv('AZURE_OPENAI_KEY')
        AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT')
        AZURE_OPENAI_DEPLOYMENT = os.getenv('AZURE_OPENAI_DEPLOYMENT')

client = AzureOpenAI(
    api_key=AZURE_OPENAI_KEY,
    api_version="2024-02-01",
    azure_endpoint=AZURE_OPENAI_ENDPOINT
)

N_RETRIES = 3

@retry(stop=stop_after_attempt(N_RETRIES), wait=wait_exponential(multiplier=1, min=4, max=70))
def generate_example(prompt, prev_examples, temperature=.5):
    messages=[
        {
            "role": "system",
            "content": f"You are generating data which will be used to train a machine learning model.\n\nYou will be given a high-level description of the model we want to train, and from that, you will generate data samples, each with a prompt/response pair.\n\nYou will do so in this format:\n```\nprompt\n-----------\n$prompt_goes_here\n-----------\n\nresponse\n-----------\n$response_goes_here\n-----------\n```\n\nOnly one prompt/response pair should be generated per turn.\n\nFor each turn, make the example slightly more complex than the last, while ensuring diversity.\n\nMake sure your samples are unique and diverse, yet high-quality and complex enough to train a well-performing model.\n\nHere is the type of model we want to train:\n`{prompt}`"
        }
    ]

    if len(prev_examples) > 0:
        if len(prev_examples) > 10:
            prev_examples = random.sample(prev_examples, 10)
        for example in prev_examples:
            messages.append({
                "role": "assistant",
                "content": example
            })

    response = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        messages=messages,
        temperature=temperature,
        max_completion_tokens=4000,
    )

    return response.choices[0].message.content

# Progressive generation with checkpoint saving in Google Drive
prev_examples = []

if os.path.exists(checkpoint_path):
    try:
        with open(checkpoint_path, 'r') as f:
            prev_examples = json.load(f)
        print(f"🔄 Resuming from checkpoint: loaded {len(prev_examples)} already-generated examples from Google Drive.")
    except Exception as e:
        print(f"Could not load checkpoint: {e}. Starting from scratch.")

if os.path.exists(train_path) and os.path.exists(test_path):
    print("✨ Found existing 'train.jsonl' and 'test.jsonl' on Google Drive! Skipping generation.")
    skip_generation = True
else:
    skip_generation = False
    start_idx = len(prev_examples)
    if start_idx < number_of_examples:
        for i in range(start_idx, number_of_examples):
            print(f'Generating example {i+1}/{number_of_examples}...')
            example = generate_example(prompt, prev_examples, temperature)
            prev_examples.append(example)
            with open(checkpoint_path, 'w') as f:
                json.dump(prev_examples, f)
        print("Generation complete!")

We also need to generate a system message.

In [ ]:
import os
import random
import json
from openai import AzureOpenAI
from tenacity import retry, stop_after_attempt, wait_exponential

try:
    # Try loading from Google Colab Secrets
    from google.colab import userdata
    AZURE_OPENAI_KEY = userdata.get('AZURE_OPENAI_KEY')
    AZURE_OPENAI_ENDPOINT = userdata.get('AZURE_OPENAI_ENDPOINT')
    AZURE_OPENAI_DEPLOYMENT = userdata.get('AZURE_OPENAI_DEPLOYMENT')
except ImportError:
    try:
        # Try loading from Kaggle Secrets
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        AZURE_OPENAI_KEY = user_secrets.get_secret('AZURE_OPENAI_KEY')
        AZURE_OPENAI_ENDPOINT = user_secrets.get_secret('AZURE_OPENAI_ENDPOINT')
        AZURE_OPENAI_DEPLOYMENT = user_secrets.get_secret('AZURE_OPENAI_DEPLOYMENT')
    except ImportError:
        # Fallback to local environment variables
        AZURE_OPENAI_KEY = os.getenv('AZURE_OPENAI_KEY')
        AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT')
        AZURE_OPENAI_DEPLOYMENT = os.getenv('AZURE_OPENAI_DEPLOYMENT')

client = AzureOpenAI(
    api_key=AZURE_OPENAI_KEY,
    api_version="2024-02-01",
    azure_endpoint=AZURE_OPENAI_ENDPOINT
)

N_RETRIES = 3

@retry(stop=stop_after_attempt(N_RETRIES), wait=wait_exponential(multiplier=1, min=4, max=70))
def generate_example(prompt, prev_examples, temperature=.5):
    messages=[
        {
            "role": "system",
            "content": f"You are generating data which will be used to train a machine learning model.\n\nYou will be given a high-level description of the model we want to train, and from that, you will generate data samples, each with a prompt/response pair.\n\nYou will do so in this format:\n```\nprompt\n-----------\n$prompt_goes_here\n-----------\n\nresponse\n-----------\n$response_goes_here\n-----------\n```\n\nOnly one prompt/response pair should be generated per turn.\n\nFor each turn, make the example slightly more complex than the last, while ensuring diversity.\n\nMake sure your samples are unique and diverse, yet high-quality and complex enough to train a well-performing model.\n\nHere is the type of model we want to train:\n`{prompt}`"
        }
    ]

    if len(prev_examples) > 0:
        if len(prev_examples) > 10:
            prev_examples = random.sample(prev_examples, 10)
        for example in prev_examples:
            messages.append({
                "role": "assistant",
                "content": example
            })

    response = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        messages=messages,
        temperature=temperature,
        max_completion_tokens=4000,
    )

    return response.choices[0].message.content

# Progressive generation with checkpoint saving in Google Drive
prev_examples = []

if os.path.exists(checkpoint_path):
    try:
        with open(checkpoint_path, 'r') as f:
            prev_examples = json.load(f)
        print(f"🔄 Resuming from checkpoint: loaded {len(prev_examples)} already-generated examples from Google Drive.")
    except Exception as e:
        print(f"Could not load checkpoint: {e}. Starting from scratch.")

if os.path.exists(train_path) and os.path.exists(test_path):
    print("✨ Found existing 'train.jsonl' and 'test.jsonl' on Google Drive! Skipping generation.")
    skip_generation = True
else:
    skip_generation = False
    start_idx = len(prev_examples)
    if start_idx < number_of_examples:
        for i in range(start_idx, number_of_examples):
            print(f'Generating example {i+1}/{number_of_examples}...')
            example = generate_example(prompt, prev_examples, temperature)
            prev_examples.append(example)
            with open(checkpoint_path, 'w') as f:
                json.dump(prev_examples, f)
        print("Generation complete!")

Now let's put our examples into a dataframe and turn them into a final pair of datasets.

In [ ]:
import pandas as pd

if os.path.exists(train_path) and os.path.exists(test_path) and skip_generation:
    # Load directly from existing files in Google Drive
    train_df = pd.read_json(train_path, lines=True)
    test_df = pd.read_json(test_path, lines=True)
    df = pd.concat([train_df, test_df], ignore_index=True)
    print(f"✨ Loaded {len(df)} examples directly from existing 'train.jsonl' and 'test.jsonl' on Google Drive.")
else:
    # Initialize lists to store prompts and responses
    prompts = []
    responses = []

    # Parse out prompts and responses from examples
    for example in prev_examples:
        split_example = example.split('-----------')
        if len(split_example) >= 4:
            prompts.append(split_example[1].strip())
            responses.append(split_example[3].strip())
        else:
            print(f"Skipping malformed example: {example[:100]}...")

    # Create a DataFrame
    df = pd.DataFrame({
        'prompt': prompts,
        'response': responses
    })

    # Remove duplicates
    df = df.drop_duplicates()
    print(f"There are {len(df)} successfully-generated examples.")

df.head()

Split into train and test sets.

In [ ]:
# Split the data into train and test sets, with 90% in the train set
if os.path.exists(train_path) and os.path.exists(test_path) and skip_generation:
    train_df = pd.read_json(train_path, lines=True)
    test_df = pd.read_json(test_path, lines=True)
    print("Re-using existing train.jsonl and test.jsonl files on Google Drive.")
else:
    train_df = df.sample(frac=0.9, random_state=42)
    test_df = df.drop(train_df.index)

    # Save the dataframes to .jsonl files in Google Drive
    train_df.to_json(train_path, orient='records', lines=True)
    test_df.to_json(test_path, orient='records', lines=True)
    print("Saved train.jsonl and test.jsonl files to Google Drive.")
    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)

# Install necessary libraries

# Define Hyperparameters

In [ ]:
model_name = "NousResearch/llama-2-7b-chat-hf" # use this if you have access to the official LLaMA 2 model "meta-llama/Llama-2-7b-chat-hf", though keep in mind you'll need to pass a Hugging Face key argument
dataset_name = "train.jsonl"
new_model = "llama-2-7b-custom"
lora_r = 64
lora_alpha = 16
lora_dropout = 0.1
use_4bit = True
bnb_4bit_compute_dtype = "float16"
bnb_4bit_quant_type = "nf4"
use_nested_quant = False
output_dir = "./results"
num_train_epochs = 1
fp16 = True
bf16 = False
per_device_train_batch_size = 2
per_device_eval_batch_size = 4
gradient_accumulation_steps = 4
gradient_checkpointing = True
max_grad_norm = 0.3
learning_rate = 2e-4
weight_decay = 0.001
optim = "paged_adamw_32bit"
lr_scheduler_type = "constant"
max_steps = -1
warmup_ratio = 0.03
group_by_length = True
save_steps = 25
logging_steps = 5
max_seq_length = 1024
packing = False
device_map = {"": 0}

#Load Datasets and Train

In [ ]:
# Load datasets
train_dataset = load_dataset('json', data_files=train_path, split="train")
valid_dataset = load_dataset('json', data_files=test_path, split="train")

# Preprocess datasets
train_dataset_mapped = train_dataset.map(lambda examples: {'text': [f'[INST] <<SYS>>\\n{system_message.strip()}\\n<</SYS>>\\n\\n' + prompt + ' [/INST] ' + response for prompt, response in zip(examples['prompt'], examples['response'])]}, batched=True)
valid_dataset_mapped = valid_dataset.map(lambda examples: {'text': [f'[INST] <<SYS>>\\n{system_message.strip()}\\n<</SYS>>\\n\\n' + prompt + ' [/INST] ' + response for prompt, response in zip(examples['prompt'], examples['response'])]}, batched=True)

compute_dtype = getattr(torch, bnb_4bit_compute_dtype)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map=device_map,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)
model.config.use_cache = False
model.config.pretraining_tp = 1
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
)
# Set training parameters
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
    report_to="all",
    evaluation_strategy="steps",
    eval_steps=5  # Evaluate every 20 steps
)
# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset_mapped,
    eval_dataset=valid_dataset_mapped,  # Pass validation dataset here
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    args=training_arguments,
    packing=packing,
)
trainer.train()
trainer.model.save_pretrained(new_model)

# Cell 4: Test the model
logging.set_verbosity(logging.CRITICAL)
prompt = f"[INST] <<SYS>>\\n{system_message}\\n<</SYS>>\\n\\nDoctor: Good morning, Mr. Smith. How are you feeling today?\\nPatient: Hi doctor, I've had a really bad headache for the past three days. It's mostly behind my eyes.\\nDoctor: Any nausea or sensitivity to light?\\nPatient: A little bit of sensitivity to light, but no nausea. [/INST]" # replace the command here with something relevant to your task
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=500)
result = pipe(prompt)
print(result[0]['generated_text'])

#Run Inference

In [ ]:
from transformers import pipeline

prompt = f"[INST] <<SYS>>\\n{system_message}\\n<</SYS>>\\n\\nDoctor: Good morning, Mr. Smith. How are you feeling today?\\nPatient: Hi doctor, I've had a really bad headache for the past three days. It's mostly behind my eyes.\\nDoctor: Any nausea or sensitivity to light?\\nPatient: A little bit of sensitivity to light, but no nausea. [/INST]" # replace the command here with something relevant to your task
num_new_tokens = 250  # change to the number of new tokens you want to generate

# Count the number of tokens in the prompt
num_prompt_tokens = len(tokenizer(prompt)['input_ids'])

# Calculate the maximum length for the generation
max_length = num_prompt_tokens + num_new_tokens

gen = pipeline('text-generation', model=model, tokenizer=tokenizer, max_length=max_length)
result = gen(prompt)
print(result[0]['generated_text'].replace(prompt, ''))

#Merge the model and store in Google Drive

In [ ]:
# Merge and save the fine-tuned model
from google.colab import drive
drive.mount('/content/drive')

model_path = "/content/drive/MyDrive/llama-2-7b-custom"  # change to your preferred path

# Reload model in FP16 and merge it with LoRA weights
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map=device_map,
)
model = PeftModel.from_pretrained(base_model, new_model)
model = model.merge_and_unload()

# Reload tokenizer to save it
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Save the merged model
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

# Load a fine-tuned model from Drive and run inference

In [ ]:
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer

drive.mount('/content/drive')

model_path = "/content/drive/MyDrive/llama-2-7b-custom"  # change to the path where your model is saved

model = AutoModelForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [ ]:
from transformers import pipeline

prompt = "What is 2 + 2?"  # change to your desired prompt
gen = pipeline('text-generation', model=model, tokenizer=tokenizer)
result = gen(prompt)
print(result[0]['generated_text'])